# Duração na distribuição (hipótese D) — resultados preliminares

No dia da **distribuição**, o modelo vê só a capa (classe, assunto, vara, município, instância) e estima uma **janela** de dias até a baixa 22.

1. Tabela com uma linha por processo.
2. **Treino 80% / teste 20%** (sorteio estratificado por grau; o teste **não entra** no treino).
3. Baseline = mediana da classe × grau no treino; modelo = P10 / P50 / P90.
4. Lista de processos do teste para escolher.

Filtro preliminar: ajuizamento desde 2020 e duração entre 1 e 10 anos (senão execuções fiscais de 2012 que só baixaram em 2025 distorcem a janela). Continua a ser uma coorte de **quem já baixou em 2025**.

In [1]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, median_absolute_error
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
if not (ROOT / "scripts" / "build_capa_duracao.py").exists():
    ROOT = Path("/workspace")

DATA = ROOT / "data" / "processos_capa_duracao.parquet"
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT.resolve())

ROOT = /workspace


## 1. Dados

In [2]:
if not DATA.exists():
    print("A construir a tabela a partir do Zenodo...")
    subprocess.check_call([sys.executable, str(ROOT / "scripts" / "build_capa_duracao.py")])

raw = pd.read_parquet(DATA)
print(f"Bruto: {len(raw):,} processos | mediana {raw['duracao_dias'].median():.0f} dias")

df = raw.loc[
    raw["dt_ajuizamento"].ge("2020-01-01")
    & raw["dt_ajuizamento"].le("2025-12-31")
    & raw["dt_baixa_22"].dt.year.eq(2025)
    & raw["duracao_dias"].between(1, 3650)
    & raw["processo_id"].notna()
    & raw["classe_codigo"].notna()
    & raw["grau"].notna()
].copy()
dups = int(df["processo_id"].duplicated().sum())
df = df.drop_duplicates("processo_id", keep="first")
print(f"Após filtro 2020–2025: {len(df):,} | mediana {df['duracao_dias'].median():.0f} dias | p90 {df['duracao_dias'].quantile(0.9):.0f} | duplicados removidos={dups}")
print(df.groupby("grau")["duracao_dias"].agg(n="count", mediana="median"))
df.head()

Bruto: 579,220 processos | mediana 523 dias


Após filtro 2020–2025: 501,411 | mediana 411 dias | p90 1397 | duplicados removidos=8
           n     mediana
grau                    
G1     93064  440.667130
G2     54266  222.216510
JE    287481  558.896701
TR     66321  119.746007
TRU      279  112.864803


,processo_id,numero_processo,grau,classe_codigo,classe_nome,assunto_codigo,assunto_nome,n_assuntos,orgao_codigo,orgao_nome,municipio_ibge,sistema_codigo,formato_codigo,dt_ajuizamento,dt_distribuicao,dt_baixa_22,inicio_fonte,duracao_dias
0,TRF2_G1_50105679020244025102,50105679020244025102,G1,12154,Execução de Título Extrajudicial,9606,Compromisso,1,12866,7ª Vara Federal de Niterói,3304557,4,1,2024-10-04 18:56:50+00:00,2024-10-04 18:56:50+00:00,2025-01-31 12:39:36+00:00,distribuicao,118.738032
1,TRF2_G1_50031210520254025101,50031210520254025101,G1,1733,Procedimento Investigatório Criminal (PIC-MP),3613,Crimes Resultante de Preconceito de Raça ou de...,1,12239,1ª Vara Federal Criminal do Rio de Janeiro,3304557,4,1,2025-01-17 16:24:07+00:00,2025-01-17 16:24:07+00:00,2025-01-27 15:04:54+00:00,distribuicao,9.944988
4,TRF2_G1_50666425520244025101,50666425520244025101,G1,7,Procedimento Comum Cível,11811,Práticas Abusivas,1,12242,19ª Vara Federal do Rio de Janeiro,3304557,4,1,2024-08-30 17:59:00+00:00,2024-08-30 17:59:00+00:00,2025-01-31 12:44:56+00:00,distribuicao,153.781898
5,TRF2_G1_50109013520214025101,50109013520214025101,G1,1118,Embargos à Execução Fiscal,6017,Dívida Ativa (Execução Fiscal),2,17760,10ª Vara de Execução Fiscal do Rio de Janeiro,3304557,4,1,2021-02-28 16:47:05+00:00,2021-02-28 16:47:05+00:00,2025-01-23 15:08:30+00:00,distribuicao,1424.931539
6,TRF2_G1_50066038620244025006,50066038620244025006,G1,120,Mandado de Segurança Cível,6138,Reajustes e Revisões Específicos,1,13794,1ª Vara Federal de Serra,3205002,4,1,2024-09-25 10:58:48+00:00,2024-09-25 10:58:48+00:00,2025-01-26 18:04:29+00:00,distribuicao,123.295613


## 2. Treino e teste

20% dos processos ficam de fora do treino (sorteio com semente 22, proporcional por grau). É desta lista que se escolhe o processo para ver previsão vs duração real.

In [3]:
CAT_COLS = ["grau", "classe_codigo", "assunto_codigo", "orgao_codigo", "municipio_ibge"]
NUM_COLS = ["n_assuntos"]
FEATURES = CAT_COLS + NUM_COLS
TARGET = "duracao_dias"
MAX_BINS = 255

train, test = train_test_split(
    df, test_size=0.20, random_state=22, stratify=df["grau"]
)
train = train.copy()
test = test.copy()

def lump_to_train(train_s: pd.Series, test_s: pd.Series, cap: int = MAX_BINS - 1) -> tuple[pd.Series, pd.Series]:
    vc = train_s.astype("string").value_counts()
    keep = set(vc.head(cap).index)
    tr = train_s.astype("string").where(train_s.astype("string").isin(keep), other="OUTROS")
    te = test_s.astype("string").where(test_s.astype("string").isin(keep), other="OUTROS")
    cats = pd.Index(tr.unique())
    return pd.Categorical(tr, categories=cats), pd.Categorical(te, categories=cats)

for col in CAT_COLS:
    train[col], test[col] = lump_to_train(train[col], test[col])

print(f"Treino: {len(train):,} | Teste: {len(test):,}")
print("IDs de teste disjuntos do treino:", set(test["processo_id"]).isdisjoint(train["processo_id"]))
print("Níveis categóricos:", {c: train[c].nunique() for c in CAT_COLS})

/tmp/ipykernel_14897/1748043314.py:19: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  return pd.Categorical(tr, categories=cats), pd.Categorical(te, categories=cats)


Treino: 401,128 | Teste: 100,283


IDs de teste disjuntos do treino: True
Níveis categóricos: {'grau': 5, 'classe_codigo': 144, 'assunto_codigo': 255, 'orgao_codigo': 255, 'municipio_ibge': 27}


## 3. Baseline e modelo (janela P10 / P50 / P90)

In [4]:
def fit_quantile(q: float) -> HistGradientBoostingRegressor:
    model = HistGradientBoostingRegressor(
        loss="quantile",
        quantile=q,
        max_depth=6,
        max_iter=250,
        learning_rate=0.06,
        max_bins=MAX_BINS,
        categorical_features=list(range(len(CAT_COLS))),
        random_state=22,
    )
    model.fit(train[FEATURES], train[TARGET])
    return model

print("A treinar P10 / P50 / P90...")
m10, m50, m90 = fit_quantile(0.10), fit_quantile(0.50), fit_quantile(0.90)

baseline_map = train.groupby(["classe_codigo", "grau"], observed=True)[TARGET].median()
global_med = float(train[TARGET].median())

def baseline_predict(frame: pd.DataFrame) -> np.ndarray:
    key = list(zip(frame["classe_codigo"], frame["grau"]))
    return np.array([baseline_map.get(k, global_med) for k in key], dtype=float)

test["pred_p10"] = np.clip(m10.predict(test[FEATURES]), 1, None)
test["pred_p50"] = np.clip(m50.predict(test[FEATURES]), 1, None)
test["pred_p90"] = np.clip(m90.predict(test[FEATURES]), 1, None)
test["pred_p90"] = np.maximum(test["pred_p90"], test["pred_p50"])
test["pred_p10"] = np.minimum(test["pred_p10"], test["pred_p50"])
test["pred_baseline"] = baseline_predict(test)
test["erro_p50"] = test[TARGET] - test["pred_p50"]
test["na_janela"] = (test[TARGET] >= test["pred_p10"]) & (test[TARGET] <= test["pred_p90"])
print("ok")

A treinar P10 / P50 / P90...


ok


In [5]:
def pinball(y, yhat, q):
    diff = y - yhat
    return float(np.mean(np.where(diff >= 0, q * diff, (q - 1) * diff)))

y = test[TARGET].to_numpy()
print("=== Teste (não usado no treino) ===")
print(f"MAE  baseline (mediana classe×grau): {mean_absolute_error(y, test['pred_baseline']):.1f} dias")
print(f"MAE  modelo P50:                     {mean_absolute_error(y, test['pred_p50']):.1f} dias")
print(f"MedAE modelo P50:                    {median_absolute_error(y, test['pred_p50']):.1f} dias")
print(
    "Pinball P10 / P50 / P90:            "
    f"{pinball(y, test['pred_p10'], 0.1):.1f} / {pinball(y, test['pred_p50'], 0.5):.1f} / {pinball(y, test['pred_p90'], 0.9):.1f}"
)
print(f"Cobertura da janela [P10, P90]:      {test['na_janela'].mean():.1%} (alvo ~80%)")
print(f"Mediana real no teste:               {np.median(y):.1f} dias")

=== Teste (não usado no treino) ===
MAE  baseline (mediana classe×grau): 334.9 dias
MAE  modelo P50:                     202.1 dias
MedAE modelo P50:                    110.6 dias
Pinball P10 / P50 / P90:            34.4 / 101.0 / 56.1
Cobertura da janela [P10, P90]:      79.2% (alvo ~80%)
Mediana real no teste:               415.0 dias


## 4. O que pesa mais na capa?

Baralha-se uma variável de cada vez no teste. Quanto mais o MAE do P50 sobe, mais essa variável importava.

In [6]:
perm = permutation_importance(
    m50,
    test[FEATURES],
    test[TARGET],
    n_repeats=8,
    random_state=22,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
imp = pd.DataFrame({
    "variavel": FEATURES,
    "aumento_mae_dias": perm.importances_mean,
    "desvio": perm.importances_std,
}).sort_values("aumento_mae_dias", ascending=False)
imp.to_csv(OUT / "importancia_capa_p50.csv", index=False)
print(imp.to_string(index=False))
print("\nSe classe_codigo liderar: o tipo de ação puxa mais o tempo típico do que a vara.")

      variavel  aumento_mae_dias   desvio
assunto_codigo        173.254746 0.392914
  orgao_codigo         84.086655 0.271726
 classe_codigo         59.287508 0.288533
          grau         14.564632 0.229700
municipio_ibge          7.961108 0.102217
    n_assuntos          1.721781 0.039383

Se classe_codigo liderar: o tipo de ação puxa mais o tempo típico do que a vara.


## 5. Lista para escolher (só teste)

Estes `processo_id` **não entraram no treino**. Lista curta abaixo; ficheiro completo (amostra gravada) em `outputs/lista_para_escolher.csv`.

In [7]:
cols = [
    "processo_id", "numero_processo", "grau", "classe_nome", "assunto_nome",
    "orgao_nome", "municipio_ibge", "dt_ajuizamento", "dt_baixa_22",
    TARGET, "pred_p10", "pred_p50", "pred_p90", "pred_baseline", "na_janela", "erro_p50",
]
holdout = test[cols].copy()
for c in [TARGET, "pred_p10", "pred_p50", "pred_p90", "pred_baseline", "erro_p50"]:
    holdout[c] = holdout[c].round(1)

holdout.to_csv(OUT / "holdout_para_escolher.csv", index=False)
(OUT / "holdout_processo_ids.txt").write_text(
    "\n".join(holdout["processo_id"].astype(str)) + "\n", encoding="utf-8"
)

rng = np.random.default_rng(22)
holdout = holdout.assign(
    faixa=pd.qcut(holdout[TARGET], q=3, labels=["curto", "tipico", "longo"], duplicates="drop")
)
lista = pd.concat(
    [
        g.sample(n=min(2, len(g)), random_state=22)
        for _, g in holdout.groupby(["faixa", "grau"], observed=True)
    ]
).reset_index(drop=True)
lista.to_csv(OUT / "lista_para_escolher.csv", index=False)
print(f"Teste completo: {len(holdout):,} IDs em outputs/holdout_para_escolher.csv")
print(f"Amostra de {len(lista)} processos (faixa × grau) — escolha um ID:\n")
lista[["processo_id", "grau", "classe_nome", "orgao_nome", "duracao_dias", "pred_p10", "pred_p50", "pred_p90", "na_janela"]]

Teste completo: 100,283 IDs em outputs/holdout_para_escolher.csv
Amostra de 28 processos (faixa × grau) — escolha um ID:



,processo_id,grau,classe_nome,orgao_nome,duracao_dias,pred_p10,pred_p50,pred_p90,na_janela
0,TRF2_G2_50172919020244020000,G2,Agravo de Instrumento,GABINETE 13,154.0,103.7,273.4,969.7,True
1,TRF2_G2_50011763220254025117,G2,Apelação Cível,GABINETE 24,73.9,107.9,185.1,1198.4,False
2,TRF2_JE_50392483920254025101,JE,Procedimento do Juizado Especial Cível,6ª Vara Federal do Rio de Janeiro,146.1,59.4,145.6,739.5,True
3,TRF2_JE_50012052720254025103,JE,Procedimento do Juizado Especial Cível,3ª Vara Federal de Campos,138.0,67.7,143.8,227.3,True
4,TRF2_G1_50083614920244025120,G1,Mandado de Segurança Cível,2ª Vara Federal de Nova Iguaçu,137.9,32.5,146.2,589.6,True
5,TRF2_G1_50119255920254025101,G1,Carta Precatória Cível,Setor de Distribuição do Rio de Janeiro (Rio B...,26.9,4.0,13.9,41.9,True
6,TRF2_TR_50045358820234025107,TR,Recurso Inominado Cível,7ª Turma Recursal - Juiz Relator 3 - RJ,239.7,102.2,301.4,849.7,True
7,TRF2_TR_50024232820234025114,TR,Recurso Inominado Cível,6ª Turma Recursal - Juiz Relator 3 - RJ,81.0,40.5,71.4,93.5,True
8,TRF2_TRU_50710892320234025101,TRU,Agravo Interno Cível,GABINETE PRESIDÊNCIA - TRU,94.0,93.3,162.8,331.7,True
9,TRF2_TRU_51283651220234025101,TRU,Agravo Interno Cível,GABINETE PRESIDÊNCIA - TRU,128.1,58.3,105.8,179.5,True


## 6. Inspecionar um processo do teste

Substitua `PROCESSO_ID` por um valor da lista.

In [8]:
PROCESSO_ID = None  # cole aqui um processo_id da lista

if PROCESSO_ID is None:
    PROCESSO_ID = lista["processo_id"].iloc[0]
    print("A usar o primeiro da amostra:", PROCESSO_ID)

row = holdout.loc[holdout["processo_id"] == PROCESSO_ID]
if row.empty:
    raise SystemExit(f"{PROCESSO_ID} não está no teste.")

r = row.iloc[0]
print(f"Processo:      {r.processo_id}")
print(f"N.º CNJ:       {r.numero_processo}")
print(f"Capa:          {r.grau} | {r.classe_nome} | {r.assunto_nome}")
print(f"Vara:          {r.orgao_nome} (IBGE {r.municipio_ibge})")
print(f"Ajuizamento:   {r.dt_ajuizamento}")
print(f"Baixa 22:      {r.dt_baixa_22}")
print(f"Duração real:  {r.duracao_dias:.1f} dias")
print(f"Baseline:      {r.pred_baseline:.1f} dias")
print(f"Janela modelo: {r.pred_p10:.1f} — {r.pred_p50:.1f} — {r.pred_p90:.1f} dias")
print(f"Na janela P10–P90? {'sim' if r.na_janela else 'não'}")

A usar o primeiro da amostra: TRF2_G2_50172919020244020000
Processo:      TRF2_G2_50172919020244020000
N.º CNJ:       50172919020244020000
Capa:          G2 | Agravo de Instrumento | Licenciamento
Vara:          GABINETE 13 (IBGE 3304557)
Ajuizamento:   2024-12-11 15:28:24+00:00
Baixa 22:      2025-05-14 15:51:12+00:00
Duração real:  154.0 dias
Baseline:      203.4 dias
Janela modelo: 103.7 — 273.4 — 969.7 dias
Na janela P10–P90? sim
